In [ ]:
!nvidia-smi

Wed Oct 13 19:58:42 2021       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.74       Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla P100-PCIE...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   47C    P0    29W / 250W |      0MiB / 16280MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import TextDataset, DataCollatorForLanguageModeling
from transformers import TrainingArguments, Trainer,  get_cosine_schedule_with_warmup
import torch
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import re


from IPython.display import clear_output
clear_output()

___________________________

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/aiijc/wiki_movie_plots_deduped (1).csv")

In [ ]:
df.Genre.value_counts()

unknown                       6083
drama                         5964
comedy                        4379
horror                        1167
action                        1098
                              ... 
action / war / adventure         1
fantasy / adventure / kids       1
biography drama                  1
south african western            1
crime thriller, neo-noir         1
Name: Genre, Length: 2265, dtype: int64

In [ ]:
df = df[df.Genre == 'comedy']
df.head()

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
7,1904,The Suburbanite,American,Wallace McCutcheon,NaN,comedy,https://en.wikipedia.org/wiki/The_Suburbanite,The film is about a family who move to the sub...
14,1907,How Brown Saw the Baseball Game,American,Unknown,Unknown,comedy,https://en.wikipedia.org/wiki/How_Brown_Saw_th...,Before heading out to a baseball game at a nea...
15,1907,Laughing Gas,American,Edwin Stanton Porter,"Bertha Regustus, Edward Boulden",comedy,https://en.wikipedia.org/wiki/Laughing_Gas_(fi...,The plot is that of a black woman going to the...
18,1908,A Calamitous Elopement,American,D.W. Griffith,"Harry Solter, Linda Arvidson",comedy,https://en.wikipedia.org/wiki/A_Calamitous_Elo...,A young couple decides to elope after being ca...
29,1910,"Hemlock Hoax, the Detective",American,Unknown,NaN,comedy,"https://en.wikipedia.org/wiki/Hemlock_Hoax,_th...",Hemlock Hoax is a detective who has little res...


In [ ]:
df = df.dropna()

In [ ]:
df.Genre.value_counts()

comedy    4347
Name: Genre, dtype: int64

In [ ]:
df.rename(columns = {'Genre':'genre'}, inplace = True)
df.genre.value_counts()

comedy    4347
Name: genre, dtype: int64

In [ ]:
bos_token = '<BOS>'
eos_token = '<EOS>'
title_token = '<TITLE>'

def build_dataset(df, path):
    f = open(path, 'w')
    data = ''
    summaries = df['Plot'].tolist()
    titles = df['Title'].tolist()
    for i in range(len(summaries)):
        summary = str(summaries[i]).strip()
        title = titles[i]
        summary = re.sub(r"\s", " ", summary)
        data += title_token + ' ' + title + ' ' + bos_token + ' ' + summary + ' ' + eos_token + '\n'
        
    f.write(data)


In [ ]:
build_dataset(df, 'train.txt')

In [ ]:
model_name = "gpt2-large"

special_tokens_dict = {'bos_token': '<BOS>', 'eos_token': '<EOS>', 'pad_token': '<PAD>'}

tokenizer = GPT2Tokenizer.from_pretrained(model_name,
                                          truncation=True,
                                          padding=True)

num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)

model = GPT2LMHeadModel.from_pretrained(model_name)

model.resize_token_embeddings(len(tokenizer))
model = model.to('cuda')

Downloading:   0%|          | 0.00/0.99M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/666 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/3.02G [00:00<?, ?B/s]

In [ ]:
learning_rate = 2e-5
batch_size = 1
num_epochs = 2

block_size = 256
train_dataset = TextDataset(tokenizer=tokenizer,
                            file_path= 'train.txt',
                            block_size=block_size,
                            overwrite_cache=True)


data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

/usr/local/lib/python3.7/dist-packages/transformers/data/datasets/language_modeling.py:58: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/master/examples/pytorch/language-modeling/run_mlm.py
  FutureWarning,


In [ ]:
seed = 42
training_args = TrainingArguments(
    output_dir='stories_output',
    overwrite_output_dir=True,
    do_train=True,
    save_steps=5000, 
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    prediction_loss_only=True,
    logging_steps=100,
    seed=seed,
    learning_rate=learning_rate
)

In [ ]:
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

In [ ]:
train_dataloader = trainer.get_train_dataloader()
num_train_steps = len(train_dataloader) * num_epochs
trainer.create_optimizer_and_scheduler(num_train_steps)

trainer.lr_scheduler = get_cosine_schedule_with_warmup(
    trainer.optimizer,
    num_warmup_steps=num_train_steps // 10,
    num_training_steps=num_train_steps
)

In [ ]:
trainer.train()

***** Running training *****
  Num examples = 7902
  Num Epochs = 2
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 1
  Gradient Accumulation steps = 1
  Total optimization steps = 15804


Step,Training Loss
100,3.415500
200,3.427600
300,3.339200
400,3.338100
500,3.347800
600,3.249600
700,3.275900
800,3.245000
900,3.312000
1000,3.256300


Saving model checkpoint to stories_output/checkpoint-5000
Configuration saved in stories_output/checkpoint-5000/config.json
Model weights saved in stories_output/checkpoint-5000/pytorch_model.bin
tokenizer config file saved in stories_output/checkpoint-5000/tokenizer_config.json
Special tokens file saved in stories_output/checkpoint-5000/special_tokens_map.json
added tokens file saved in stories_output/checkpoint-5000/added_tokens.json
Saving model checkpoint to stories_output/checkpoint-10000
Configuration saved in stories_output/checkpoint-10000/config.json
Model weights saved in stories_output/checkpoint-10000/pytorch_model.bin
tokenizer config file saved in stories_output/checkpoint-10000/tokenizer_config.json
Special tokens file saved in stories_output/checkpoint-10000/special_tokens_map.json
added tokens file saved in stories_output/checkpoint-10000/added_tokens.json
Saving model checkpoint to stories_output/checkpoint-15000
Configuration saved in stories_output/checkpoint-15000/

TrainOutput(global_step=15804, training_loss=3.0086452349684745, metrics={'train_runtime': 6893.1748, 'train_samples_per_second': 2.293, 'train_steps_per_second': 2.293, 'total_flos': 1.71961372901376e+16, 'train_loss': 3.0086452349684745, 'epoch': 2.0})

In [ ]:
model.eval()

def generate(prompt):
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    input_ids = input_ids.to('cuda') 
    eos_id = tokenizer.encode(eos_token)[0]

    sample_outputs = model.generate(
        input_ids, 
        max_length=900, 
        min_length=1000,
        do_sample=True, 
        top_k=0,
        top_p=0.92, 
        temperature=0.85,
        eos_token_id=eos_id,
        num_return_sequences=3,
        pad_token_id=tokenizer.eos_token_id,
        repition_penalty=1.5)
    for i, sample_output in enumerate(sample_outputs):
          print("{}: {}\n----------------\n".format(i+1, tokenizer.decode(sample_output, skip_special_tokens=True)))

In [ ]:
title = 'The show'

prompt = title_token + ' ' + title + ' ' + bos_token

generate(prompt)

1: <TITLE> The showLa Rueff (Tony Curtis) is a young man who dreams of being an actress. He works as a waitress in an obscure diner. One day, he meets a beautiful young man, Sabine (Joan Blondell), and falls for her. After she leaves, he goes to a nightclub to find her. There he meets an uncredited but excellent actor, Bob (Brian Doyle-Murray), who plays a role in a film that is set in the same diner where La Rueff works. Sabine introduces La Rueff to her family, who are all actors. Sabine's sister, Marissa (Joan Blondell), has been a minor Hollywood star for several years. La Rueff has been getting advice from Marissa's agent and learns that Marissa wants to marry a man named Rudy (Lance Kinsey), who is a porn star. Rudy is not in the mood for a relationship, but La Rueff confides in Sabine that he is going to New York City to audition for a film.  La Rueff goes to New York City, auditioning for a part in a pornographic film. He is only hired for a cameo in the film. During the auditi